# 2. Applying neurons and writing array expressions

CS 301 · Unit II essentials practice

**Why we do this.** A layer applies several scoring rules to every input. Array operations let us express those rules clearly and check what each output means.

**By the end, you should be able to:**

- Implement a layer and interpret its rows, columns, and biases.
- Apply softmax and argmax and distinguish them from sigmoid.
- Use axes, broadcasting, and batch dimensions to write correct array code.

Estimated working time: **40–50 minutes**, including written responses.

## Working through this notebook

Save your own copy. Run the setup, then work in order. Before each exercise, write a short prediction. Replace `...` in the exercise cells, run the checks, and explain the result in your own words. A passing check verifies the calculation; the written response shows how you interpret it.

Each notebook runs independently. No results from another notebook are needed. The examples use modest arrays; a GPU is optional in Notebook 3.

**Name:** …


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=4, suppress=True)

def ready(*values):
    assert all(v is not Ellipsis and v is not None for v in values), "Replace every ... in the exercise before running its check."

def heatmap(A, title, labels=None, limits=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    scale = {} if limits is None else {"vmin": limits[0], "vmax": limits[1]}
    im = ax.imshow(A, cmap="coolwarm", aspect="auto", **scale)
    ax.set_title(title)
    if labels is not None:
        ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        ax.set_yticks(range(len(labels)), labels)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()

from urllib.request import urlopen
from PIL import Image
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/images/"
images = pd.read_csv(SITE + "images.csv", index_col="number")
info = images.loc[1]
with urlopen(SITE + info["file"]) as stream:
    image = np.asarray(Image.open(stream).convert("RGB"), dtype=float)[::4, ::4] / 255
print(info["title"], "—", info["creator"])
print("Source:", info["source"])
H, width, channels = image.shape
X = image.reshape(-1, 3)
# Fixed reference colours make the effect of changing a neuron easy to inspect.
C = np.array([[.1, .1, .1], [.8, .2, .2], [.2, .7, .2], [.2, .3, .8], [.9, .9, .8]])
W = C.copy()
bias = -.5 * (C ** 2).sum(axis=1)
plt.imshow(image); plt.title("Input image"); plt.axis("off"); plt.show()
print("X:", X.shape, "W:", W.shape, "bias:", bias.shape)


## 1. Compute one score for every point and neuron

Each row of `X` is an RGB pixel. Each row of `W` contains one neuron's weights; `bias[j]` belongs to neuron j.

For one point x, neuron j computes a dot product plus its bias. For all points and neurons, the output has one row per point and one column per neuron. Implement this with one `@` and broadcasting, without loops.

These weights use reference colours from Meeting 11: the highest-scoring neuron identifies the nearest reference colour. The weights are supplied; this notebook does not train them.


**Predict before running.** Give the output shape. Explain what entry `[i, j]` will mean and which output entries receive `bias[j]`.

*Write your prediction here.*


In [ ]:
def layer(X, W, b):
    return ...

S = layer(X, W, bias)


In [ ]:
ready(S)
assert S.shape == (len(X), len(W)), "Need one score for each point and neuron."
for i in [0, len(X)//2, len(X)-1]:
    for j in range(len(W)):
        assert np.isclose(S[i, j], sum(X[i, q]*W[j, q] for q in range(3)) + bias[j]), "Check transpose and bias alignment."
rng = np.random.default_rng(7)
for n, d, k in [(7, 4, 2), (2, 3, 3)]:
    xx, ww, bb = rng.normal(size=(n,d)), rng.normal(size=(k,d)), rng.normal(size=k)
    assert np.allclose(layer(xx,ww,bb), xx @ ww.T + bb), "Your function must work with other shapes and weights."
winner = S.argmax(axis=1)
plt.imshow(C[winner].reshape(H, width, 3)); plt.title("Nearest reference colour"); plt.axis("off"); plt.show()
print("Check passed.")


**Explain.** Increase the bias of neuron 1 by 0.1 in a separate calculation. Predict and check which score entries change. What can happen to the winning neuron for a pixel?

*Write your response here.*


In [ ]:
# Add your calculation here. Preserve the original bias and S for later exercises.


## 2. Diagnose a transpose error

A shape-compatible expression can still compute the wrong quantities. Below, the number of neurons equals the number of input features, so `X_small @ W_square` runs. Compare it with the definition of a neuron's score and repair the expression.


In [ ]:
X_small = np.array([[1., 2., 0.], [0., 1., 3.]])
W_square = np.array([[1., 2., 3.], [0., 1., 2.], [4., 0., 1.]])
b_small = np.array([.5, -.5, 1.])
wrong_scores = X_small @ W_square + b_small
print("The expression runs and returns shape", wrong_scores.shape)


**Predict before running.** In the incorrect expression, does output column j use row j or column j of `W_square`? Which does the neuron definition require?

*Write your prediction here.*


In [ ]:
fixed_scores = ...


In [ ]:
ready(fixed_scores)
expected = np.array([[np.dot(x,w)+bb for w,bb in zip(W_square,b_small)] for x in X_small])
assert np.allclose(fixed_scores, expected), "Use one weight row per neuron."
assert not np.allclose(wrong_scores, expected), "The supplied example should expose the error."
display(pd.DataFrame({"wrong, first point": wrong_scores[0], "fixed, first point": fixed_scores[0]}))
print("Check passed.")


**Explain.** Why did the shape check miss this error? Propose a small test input that would reveal it.

*Write your response here.*


## 3. Turn each row of scores into probabilities

Softmax exponentiates scores and divides by their sum **within each row**. For numerical stability, first subtract the row maximum. Subtracting the same number from every score in a row leaves its softmax unchanged.

Implement the three steps with `axis=1` and `keepdims=True`. Do not call a library softmax. Then use `argmax` to select one neuron per point.


**Predict before running.** Which axis must sum to one? Should softmax change which neuron wins for a point?

*Write your prediction here.*


In [ ]:
def softmax_rows(scores):
    shifted = ...
    exponentials = ...
    return ...

P = softmax_rows(S / .1)
winners = ...


In [ ]:
ready(P, winners)
assert P.shape == S.shape and np.all(np.isfinite(P)), "Keep the score shape and avoid overflow."
assert np.all(P >= 0) and np.allclose(P.sum(axis=1), 1), "Each point needs its own probability distribution."
assert np.array_equal(winners, S.argmax(axis=1)), "Check the axis for argmax."
test = np.array([[1.,2.,3.], [1000.,1001.,1002.]])
out = softmax_rows(test)
reference = np.exp(np.array([-2.,-1.,0.])); reference /= reference.sum()
assert np.allclose(out, np.tile(reference,(2,1))), "Subtract each row's maximum before exponentiating."
print("Check passed.")
pixel = len(X)//2
plt.bar(np.arange(len(W)), P[pixel]); plt.xlabel("Neuron"); plt.ylabel("Probability"); plt.show()
# Sigmoid operates on entries independently.
sigmoid = 1 / (1 + np.exp(-S[pixel]))
print("Sigmoid values for this pixel:", sigmoid, "sum:", sigmoid.sum())


**Explain.** Compare sigmoid and softmax for this pixel. Why does only one of them enforce a row sum of one? Try temperatures 1 and 0.05: what changes, and what stays the same?

*Write your response here.*


## 4. Write and test two kinds of standardization

An `axis` identifies the dimension being reduced. `axis=0` combines rows and leaves one value per column. `axis=1` combines columns and leaves one value per row. `keepdims=True` retains a length-one dimension for broadcasting.

Write `standardize(X, axis)` for either axis. Use the population standard deviation. The test arrays contain no constant row or column.


**Predict before running.** For a `(5, 3)` array, give the shapes of means along axis 0 and axis 1, both with `keepdims=True`. Which standardization is appropriate when columns represent different measured features?

*Write your prediction here.*


In [ ]:
def standardize(X, axis):
    return ...


In [ ]:
test = np.array([[1.,4.,8.], [2.,7.,3.], [4.,1.,6.], [9.,2.,1.], [6.,8.,2.]])
for axis in [0,1]:
    result = standardize(test, axis)
    ready(result)
    assert result.shape == test.shape
    assert np.allclose(result.mean(axis=axis), 0), "Check the axis used for the mean."
    assert np.allclose(result.std(axis=axis), 1), "Check the axis and broadcasting of the standard deviation."
print("Column-standardized:", standardize(test,0), sep="\n")
print("Row-standardized:", standardize(test,1), sep="\n")
print("Check passed.")


**Explain.** Why do the two functions produce different arrays? Explain why `(5,)` row means cannot generally be subtracted from a `(5, 3)` array, and why `(5, 1)` works.

*Write your response here.*


## 5. Apply the layer to a batch of images

`batch` has shape `(N, H, width, 3)`. NumPy's `@` uses the last two dimensions as a matrix and treats earlier dimensions as batch dimensions. A weight matrix can therefore be applied to every pixel with one `@`, without reshaping.

Implement `batch_scores` and return one score per image, pixel, and neuron. Then compute the winning neuron at each pixel, reducing only the last axis.


In [ ]:
batch = np.stack([image, image[:, ::-1], np.clip(image * .7, 0, 1)])
print("Batch shape:", batch.shape)


**Predict before running.** Write the expected shapes of the score array and the winner array. What would `*` do in place of `@`? Would it express the same operation?

*Write your prediction here.*


In [ ]:
def batch_scores(batch, W, b):
    return ...

batch_S = batch_scores(batch, W, bias)
batch_winners = ...


In [ ]:
ready(batch_S, batch_winners)
assert batch_S.shape == batch.shape[:-1] + (len(W),)
assert batch_winners.shape == batch.shape[:-1], "Reduce the neuron dimension only."
for i in range(len(batch)):
    expected = layer(batch[i].reshape(-1,3), W, bias).reshape(H,width,-1)
    assert np.allclose(batch_S[i], expected), "Batch and single-image calculations must agree."
assert np.array_equal(batch_winners, batch_S.argmax(axis=-1))
fig, axes = plt.subplots(1,3,figsize=(12,4))
for i,ax in enumerate(axes):
    ax.imshow(C[batch_winners[i]]); ax.set_title(f"Image {i}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("Check passed.")


**Explain.** Explain why a single weight matrix can be used for every image. Describe one use of broadcasting and one reduction in your solution.

*Write your response here.*


## Final check

Restart the runtime and run your completed notebook from the top. Check that every exercise passes, your figures appear, and every prediction and explanation is filled in. Save the notebook with its outputs.

**Source activities:** Meeting 11 neuron and image activities, Meeting 10 image arrays, and vectorization homework Parts A and C. This practice version reorganizes those activities in the order of the Meeting 15 essentials deck.
